# insurance-distill

**Distil GBM models into multiplicative GLM factor tables for rating engines.**

This notebook shows the core workflow in under two minutes: train a GBM on synthetic motor frequency data, distil it into GLM factor tables, and inspect the validation metrics.

The problem this solves: your CatBoost model outperforms your GLM on Gini, but Radar, Emblem, and every other UK personal lines rating engine need factor tables - not a black box. This library fits a Poisson GLM on the GBM's predictions (pseudo-predictions) and exports the result as multiplicative relativities.

In [ ]:
!pip install -q insurance-distill polars numpy scikit-learn glum

## Synthetic motor data

2,000 policies. Driver age and vehicle value drive frequency non-linearly; NCD years applies a discount. This structure means the GBM will outperform a linear GLM - which is the scenario distillation is designed for.

In [ ]:
import numpy as np
import polars as pl
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(42)
n = 2_000

driver_age    = rng.uniform(18, 80, n)
vehicle_value = rng.uniform(5_000, 60_000, n)
ncd_years     = rng.integers(0, 11, n).astype(float)
exposure      = rng.uniform(0.1, 1.0, n)

# Non-linear frequency: young driver surcharge, vehicle value effect, NCD discount
log_mu = (
    -3.0
    - 0.04 * np.clip(driver_age - 25, 0, None)
    + 0.01 * np.clip(25 - driver_age, 0, None)
    + 0.00002 * vehicle_value
    - 0.08 * ncd_years
)
mu = exposure * np.exp(log_mu)
y  = rng.poisson(mu).astype(float)

X_train = pl.DataFrame({
    "driver_age":    driver_age,
    "vehicle_value": vehicle_value,
    "ncd_years":     ncd_years,
})

print(f"Policies: {n:,}  |  Claim count: {int(y.sum())}  |  Avg frequency: {y.sum()/exposure.sum():.3f}")
X_train.head()

## Fit a GBM

We use sklearn's GradientBoostingRegressor here so the notebook has no CatBoost dependency. In production, pass your fitted CatBoost model directly - `SurrogateGLM` accepts any scikit-learn-compatible model with a `predict` method.

In [ ]:
# Fit on claim rate (claims / exposure) with exposure as sample weight
rate = np.clip(y / exposure, 0, None)

gbm = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
)
gbm.fit(X_train.to_numpy(), rate, sample_weight=exposure)

print(f"GBM trained. Feature importances:")
for feat, imp in zip(X_train.columns, gbm.feature_importances_):
    print(f"  {feat:20s}: {imp:.3f}")

## Distil to GLM factor tables

`SurrogateGLM` takes the fitted GBM, generates its predictions (pseudo-predictions), bins the continuous variables using a CART decision tree to find statistically meaningful cut-points, then fits a Poisson log-link GLM on those binned features.

In [ ]:
from insurance_distill import SurrogateGLM

surrogate = SurrogateGLM(
    model=gbm,
    X_train=X_train,
    y_train=y,
    exposure=exposure,
    family="poisson",
)

surrogate.fit(
    features=["driver_age", "vehicle_value", "ncd_years"],
    max_bins=8,
    binning_method="tree",
    method_overrides={"ncd_years": "isotonic"},   # NCD is monotone: use isotonic regression
)

print("Fit complete.")

## Validation report

The report answers: how much of the GBM's discrimination does the GLM retain?

- **Gini ratio** above 0.90 is generally acceptable; above 0.95 is excellent
- **Max segment deviation** - the most operationally relevant check. If the GLM is within 5% in every cell, the factor tables are faithful
- **Deviance ratio** - analogous to R-squared for GLMs

In [ ]:
report = surrogate.report()
print(report.metrics.summary())
print()
print("Double-lift chart (GBM vs GLM by decile):")
print(report.lift_chart)

## Factor tables

Each factor table has three columns: `level` (the bin label), `log_coefficient` (the raw GLM coefficient), and `relativity` (the multiplicative factor = exp(coefficient)). The base level always has `relativity = 1.0`. These tables load directly into Radar or Emblem.

In [ ]:
print("-- driver_age factor table --")
print(surrogate.factor_table("driver_age"))
print()
print("-- ncd_years factor table --")
print(surrogate.factor_table("ncd_years"))

## Export to CSV

One CSV file per variable. The `prefix` argument lets you namespace files by model type - for example `motor_freq_driver_age.csv`. Each file matches the factor table format expected by Radar and Emblem.

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    written = surrogate.export_csv(tmpdir, prefix="motor_freq_")
    for path in written:
        print(os.path.basename(path))

## Next steps

- **Interaction terms**: pass `interaction_pairs=[("driver_age", "ncd_years")]` to `fit()` to add cross-variable terms
- **Categorical features**: pass `categorical_features=["region"]` for string columns - no binning needed
- **Severity models**: set `family="gamma"` for a Gamma GLM on claim severity
- **Regularisation**: set `alpha=0.01` in the `SurrogateGLM` constructor to apply L2 regularisation if the factor tables are noisy

**GitHub:** https://github.com/burning-cost/insurance-distill  
**PyPI:** https://pypi.org/project/insurance-distill/